# Assignment Beta — Question 1: SaafHawa (Google Colab)
**Student ID:** 22P-9288  •  **Evidence notebook:** numerical analysis + live API latency measurements.

**Run top to bottom.** Notebook works on a CPU Colab runtime. The OpenAQ key is needed only for the live air-quality tests; Open-Meteo needs no key. For the *two different calendar days* requirement, mount Google Drive to persist the CSV; execute the timing section on two different Karachi dates.

**Integrity:** Public provider prices/quotas are snapshots and may change. A GPU memory model does not establish actual throughput. Failed API calls are not counted as successful measurements. This notebook **does not call an LLM API**, because Q1 requires a comparison/estimation, not running all 12 models.

In [ ]:
%pip -q install numpy pandas matplotlib requests nltk
import os, sys, json, datetime
from pathlib import Path
print('Python:',sys.version.split()[0]); print('UTC:', datetime.datetime.now(datetime.timezone.utc).isoformat())

## 1. Persistent storage (recommended)
If on Colab, mount Drive so your `timing_log.csv` survives runtime restarts and can cover two dates. If Drive access is declined, the notebook uses `/content/Assignment_Beta_22P-9288` for this session only.

In [ ]:
from pathlib import Path
import os
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ROOT=Path('/content/drive/MyDrive/Assignment_Beta_22P-9288')
except Exception as e:
    print('Drive unavailable; temporary runtime directory:',type(e).__name__)
    ROOT=Path('/content/Assignment_Beta_22P-9288')
for name in ['code','data','results','figures']:(ROOT/name).mkdir(parents=True,exist_ok=True)
os.chdir(ROOT)
print('Working directory:',ROOT)

## 2. Load research inputs and computation scripts
The comparison-data file is included for reproducibility. It is **not automatically revalidated against live prices**; check quoted sources before submission. The calculation script uses those values. The listed Lambda GPU price is a **scenario**: Lambda indicates its listed rates take effect **November 16, 2026**; do not label it the verified November 11 billable price.

In [ ]:
from pathlib import Path
Path('data/model_research.json').parent.mkdir(parents=True,exist_ok=True)
Path('data/model_research.json').write_text('{\n  "as_of": "2026-11-10",\n  "notes": "USD per million text tokens. Quoted provider endpoints and speed observations are not universal guarantees; UNKNOWN denotes unverified values. Warm-cache model assumes 1500-token prefix served by cache on all calls in steady state; first cache write charged separately in sensitivity notes. Google explicit cache storage excluded without a grounded retention period. Open-weight provider caches are not assumed absent unless clearly documented for the chosen cheapest endpoint.",\n  "models": [\n    {\n      "name": "GPT-5.6 Luna",\n      "kind": "closed",\n      "developer": "OpenAI",\n      "endpoint": "gpt-5.6-luna",\n      "parameters": "Not disclosed",\n      "context": 1050000,\n      "max_output": 128000,\n      "input": 0.2,\n      "output": 1.2,\n      "cached": 0.02,\n      "cache_min": 1024,\n      "cache_write": 0.25,\n      "cache_lifetime": "30 min minimum (paid write)",\n      "rpm": 5000,\n      "tpm": 2000000,\n      "rate_kind": "combined TPM; Build (new paid)",\n      "ttft": null,\n      "tps": null,\n      "benchmark_tool": "Not verified for exact revision",\n      "benchmark_other": "Not verified for exact revision",\n      "native_tools": "Yes",\n      "json": "Yes",\n      "reasoning": "none",\n      "license": "Proprietary; self-hosting no",\n      "source": "https://developers.openai.com/api/docs/models/gpt-5.6-luna",\n      "rate_source": "https://developers.openai.com/api/docs/guides/rate-limits",\n      "provider": "OpenAI"\n    },\n    {\n      "name": "GPT-5.6 Terra",\n      "kind": "closed",\n      "developer": "OpenAI",\n      "endpoint": "gpt-5.6-terra",\n      "parameters": "Not disclosed",\n      "context": 1050000,\n      "max_output": 128000,\n      "input": 2.0,\n      "output": 12.0,\n      "cached": 0.2,\n      "cache_min": 1024,\n      "cache_write": 2.5,\n      "cache_lifetime": "30 min minimum (paid write)",\n      "rpm": 5000,\n      "tpm": 1000000,\n      "rate_kind": "combined TPM; Build (new paid)",\n      "ttft": null,\n      "tps": null,\n      "benchmark_tool": "Not verified for exact revision",\n      "benchmark_other": "Not verified for exact revision",\n      "native_tools": "Yes",\n      "json": "Yes",\n      "reasoning": "none",\n      "license": "Proprietary; self-hosting no",\n      "source": "https://developers.openai.com/api/docs/models/gpt-5.6-terra",\n      "rate_source": "https://developers.openai.com/api/docs/guides/rate-limits",\n      "provider": "OpenAI"\n    },\n    {\n      "name": "Claude Haiku 5.5",\n      "kind": "closed",\n      "developer": "Anthropic",\n      "endpoint": "claude-haiku-5-5",\n      "parameters": "Not disclosed",\n      "context": 1000000,\n      "max_output": 128000,\n      "input": 0.1,\n      "output": 0.5,\n      "cached": 0.01,\n      "cache_min": 512,\n      "cache_write": 0.125,\n      "cache_lifetime": "5 min write; optional 1 h at $0.20/M",\n      "rpm": 4000,\n      "tpm": 2000000,\n      "out_tpm": 400000,\n      "rate_kind": "2m uncached input TPM + 400k output TPM; Start tier; $500/month cap",\n      "ttft": null,\n      "tps": null,\n      "benchmark_tool": "Not verified for exact revision",\n      "benchmark_other": "Not verified for exact revision",\n      "native_tools": "Yes",\n      "json": "Yes (strict via API)",\n      "reasoning": "adaptive; minimum effort/between-tools setting subject to model API",\n      "license": "Proprietary; self-hosting no",\n      "source": "https://platform.claude.com/docs/en/models/haiku-5-5/overview",\n      "rate_source": "https://platform.claude.com/docs/en/api/rate-limits",\n      "provider": "Anthropic"\n    },\n    {\n      "name": "Claude Sonnet 5.5",\n      "kind": "closed",\n      "developer": "Anthropic",\n      "endpoint": "claude-sonnet-5-5",\n      "parameters": "Not disclosed",\n      "context": 1000000,\n      "max_output": 128000,\n      "input": 2.0,\n      "output": 10.0,\n      "cached": 0.1,\n      "cache_min": 512,\n      "cache_write": 2.5,\n      "cache_lifetime": "5 min write; optional 1 h at $4.00/M",\n      "rpm": 4000,\n      "tpm": 2000000,\n      "out_tpm": 400000,\n      "rate_kind": "2m uncached input TPM + 400k output TPM; Start tier; $500/month cap",\n      "ttft": null,\n      "tps": null,\n      "benchmark_tool": "Not verified for exact revision",\n      "benchmark_other": "Not verified for exact revision",\n      "native_tools": "Yes",\n      "json": "Yes (strict via API)",\n      "reasoning": "between_tools (minimum up-front)",\n      "license": "Proprietary; self-hosting no",\n      "source": "https://platform.claude.com/docs/en/models/sonnet-5-5/overview",\n      "rate_source": "https://platform.claude.com/docs/en/api/rate-limits",\n      "provider": "Anthropic"\n    },\n    {\n      "name": "Gemini 3.5 Flash-Lite",\n      "kind": "closed",\n      "developer": "Google",\n      "endpoint": "gemini-3.5-flash-lite",\n      "parameters": "Not disclosed",\n      "context": 1048576,\n      "max_output": 65536,\n      "input": 0.3,\n      "output": 2.5,\n      "cached": 0.03,\n      "cache_min": null,\n      "cache_write": null,\n      "cache_lifetime": "Explicit cache storage $1/Mtok-hour; minimum 1,500 prefix eligibility not verified",\n      "rpm": null,\n      "tpm": null,\n      "rate_kind": "Tier 1 project-specific RPM/TPM in Google AI Studio; $10 per 10min rolling spend cap may apply",\n      "ttft": null,\n      "tps": null,\n      "benchmark_tool": "Not verified for exact new revision (do not reuse Gemini 2.5 BFCL score)",\n      "benchmark_other": "Not verified exact-model",\n      "native_tools": "Yes",\n      "json": "Yes",\n      "reasoning": "thinking control supported; disable/lowest setting must be tested; billed output includes thinking",\n      "license": "Proprietary; self-hosting no",\n      "source": "https://ai.google.dev/gemini-api/docs/models/gemini-3.5-flash-lite",\n      "rate_source": "https://ai.google.dev/gemini-api/docs/rate-limits",\n      "provider": "Google"\n    },\n    {\n      "name": "Grok 4.3",\n      "kind": "closed",\n      "developer": "xAI",\n      "endpoint": "grok-4.3",\n      "parameters": "Not disclosed",\n      "context": 1000000,\n      "max_output": null,\n      "input": 1.25,\n      "output": 2.5,\n      "cached": 0.2,\n      "cache_min": null,\n      "cache_write": null,\n      "cache_lifetime": "Automatic cache; minimum and lifetime unverified",\n      "rpm": 2220,\n      "tpm": 10000000,\n      "rate_kind": "T0 37 RPS × 60, 10M TPM; T0 first accessible tier, paid tier availability varies",\n      "ttft": null,\n      "tps": null,\n      "benchmark_tool": "Not verified for exact revision",\n      "benchmark_other": "Not verified for exact revision",\n      "native_tools": "Yes",\n      "json": "Yes",\n      "reasoning": "none",\n      "license": "Proprietary; self-hosting no",\n      "source": "https://docs.x.ai/developers/pricing",\n      "rate_source": "https://docs.x.ai/developers/rate-limits",\n      "provider": "xAI"\n    },\n    {\n      "name": "Qwen3-30B-A3B Instruct 2507",\n      "kind": "open-weight",\n      "developer": "Qwen",\n      "endpoint": "qwen/qwen3-30b-a3b-instruct-2507",\n      "parameters": "30.5B total / 3.3B active MoE",\n      "context": 262144,\n      "max_output": 32000,\n      "input": 0.04815,\n      "output": 0.1931,\n      "cached": null,\n      "cache_min": null,\n      "cache_write": null,\n      "cache_lifetime": "No verified discounted-cache quote at cheapest endpoint",\n      "rpm": null,\n      "tpm": null,\n      "rate_kind": "Provider-specific; requires provider account/contract",\n      "ttft": 1.73,\n      "tps": 23,\n      "benchmark_tool": "BFCL V4 41.39% (FC)",\n      "benchmark_other": "Not verified exact-model",\n      "native_tools": "Yes on listed endpoint",\n      "json": "Yes on listed endpoint",\n      "reasoning": "non-thinking",\n      "license": "Apache-2.0; commercial self-host allowed",\n      "source": "https://openrouter.ai/qwen/qwen3-30b-a3b-instruct-2507",\n      "rate_source": "https://openrouter.ai/qwen/qwen3-30b-a3b-instruct-2507",\n      "provider": "StreamLake (temporary 55% discount)"\n    },\n    {\n      "name": "Qwen2.5-7B Instruct",\n      "kind": "open-weight",\n      "developer": "Qwen",\n      "endpoint": "qwen/qwen-2.5-7b-instruct",\n      "parameters": "7.62B dense",\n      "context": 32768,\n      "max_output": 32768,\n      "input": 0.1,\n      "output": 0.2,\n      "cached": null,\n      "cache_min": null,\n      "cache_write": null,\n      "cache_lifetime": "Not advertised on priced endpoint",\n      "rpm": null,\n      "tpm": null,\n      "rate_kind": "Phala/OpenRouter provider-specific, not published",\n      "ttft": 0.18,\n      "tps": 145,\n      "benchmark_tool": "Not verified exact-model",\n      "benchmark_other": "Not verified exact-model",\n      "native_tools": "Yes via selected gateway",\n      "json": "Yes via selected gateway",\n      "reasoning": "non-reasoning",\n      "license": "Apache-2.0 at official Qwen Hugging Face card; router displays historical Tongyi link (check weights source)",\n      "source": "https://openrouter.ai/qwen/qwen-2.5-7b-instruct",\n      "rate_source": "https://openrouter.ai/qwen/qwen-2.5-7b-instruct",\n      "provider": "Phala"\n    },\n    {\n      "name": "Llama 3.3 70B Instruct",\n      "kind": "open-weight",\n      "developer": "Meta",\n      "endpoint": "meta-llama/llama-3.3-70b-instruct",\n      "parameters": "70B dense",\n      "context": 131072,\n      "max_output": 16384,\n      "input": 0.1,\n      "output": 0.32,\n      "cached": null,\n      "cache_min": null,\n      "cache_write": null,\n      "cache_lifetime": "Not verified at cheapest endpoint",\n      "rpm": null,\n      "tpm": null,\n      "rate_kind": "OpenRouter provider-specific",\n      "ttft": null,\n      "tps": null,\n      "benchmark_tool": "BFCL V4 31.90% (FC)",\n      "benchmark_other": "Not verified exact-model",\n      "native_tools": "Yes at 7/10 endpoints",\n      "json": "Yes at eligible endpoints",\n      "reasoning": "non-reasoning",\n      "license": "Meta Llama 3.3 Community License; commercial subject to terms",\n      "source": "https://openrouter.ai/meta-llama/llama-3.3-70b-instruct",\n      "rate_source": "https://openrouter.ai/meta-llama/llama-3.3-70b-instruct",\n      "provider": "Lowest posted route (verify exact active endpoint)"\n    },\n    {\n      "name": "DeepSeek V3.2 Exp",\n      "kind": "open-weight",\n      "developer": "DeepSeek",\n      "endpoint": "deepseek/deepseek-v3.2-exp",\n      "parameters": "671B total / 37B active MoE (verify at weights card)",\n      "context": 163840,\n      "max_output": 163840,\n      "input": 0.27,\n      "output": 0.41,\n      "cached": null,\n      "cache_min": null,\n      "cache_write": null,\n      "cache_lifetime": "No quoted discounted-cache rate",\n      "rpm": null,\n      "tpm": null,\n      "rate_kind": "OpenRouter provider-specific",\n      "ttft": null,\n      "tps": null,\n      "benchmark_tool": "BFCL V4 54.12% FC; non-thinking setting",\n      "benchmark_other": "GPQA Diamond 73.8% (OpenRouter/Artificial Analysis)",\n      "native_tools": "Yes via OpenRouter",\n      "json": "Yes via OpenRouter",\n      "reasoning": "disabled / non-thinking",\n      "license": "MIT; commercial self-host permitted",\n      "source": "https://openrouter.ai/deepseek/deepseek-v3.2-exp",\n      "rate_source": "https://openrouter.ai/deepseek/deepseek-v3.2-exp",\n      "provider": "Lowest listed OpenRouter route"\n    },\n    {\n      "name": "Kimi K2 0905",\n      "kind": "open-weight",\n      "developer": "Moonshot AI",\n      "endpoint": "moonshotai/kimi-k2-0905",\n      "parameters": "1T total / 32B active MoE",\n      "context": 262144,\n      "max_output": null,\n      "input": 0.6,\n      "output": 2.5,\n      "cached": null,\n      "cache_min": null,\n      "cache_write": null,\n      "cache_lifetime": "No verified discount",\n      "rpm": null,\n      "tpm": null,\n      "rate_kind": "NovitaAI/OpenRouter provider-specific",\n      "ttft": 0.64,\n      "tps": 41,\n      "benchmark_tool": "τ²-Bench Telecom 73.4% (Artificial Analysis)",\n      "benchmark_other": "MMLU-Pro 81.9%; GPQA Diamond 76.7%",\n      "native_tools": "Yes",\n      "json": "Endpoint-dependent; verify before deployment",\n      "reasoning": "instruct / non-thinking",\n      "license": "Modified MIT license; commercial conditions require review",\n      "source": "https://openrouter.ai/moonshotai/kimi-k2-0905",\n      "rate_source": "https://openrouter.ai/moonshotai/kimi-k2-0905",\n      "provider": "NovitaAI"\n    },\n    {\n      "name": "Mistral Small 3.2 24B",\n      "kind": "open-weight",\n      "developer": "Mistral AI",\n      "endpoint": "mistralai/mistral-small-3.2-24b-instruct",\n      "parameters": "24B dense",\n      "context": 262144,\n      "max_output": null,\n      "input": 0.075,\n      "output": 0.2,\n      "cached": null,\n      "cache_min": null,\n      "cache_write": null,\n      "cache_lifetime": "Lowest price DeepInfra has no cache-read quote",\n      "rpm": null,\n      "tpm": null,\n      "rate_kind": "DeepInfra/OpenRouter provider-specific",\n      "ttft": 0.34,\n      "tps": 52,\n      "benchmark_tool": "BFCL V4 Mistral-small-2506 FC 37.15% (verify exact snapshot mapping)",\n      "benchmark_other": "Not verified exact-model",\n      "native_tools": "Yes",\n      "json": "Yes",\n      "reasoning": "non-reasoning",\n      "license": "Apache-2.0 (weights card); commercial self-host allowed",\n      "source": "https://openrouter.ai/mistralai/mistral-small-3.2-24b-instruct",\n      "rate_source": "https://openrouter.ai/mistralai/mistral-small-3.2-24b-instruct",\n      "provider": "DeepInfra"\n    }\n  ]\n}', encoding="utf-8")
print("Saved", 'data/model_research.json')

In [ ]:
from pathlib import Path
Path('data/advisories.json').parent.mkdir(parents=True,exist_ok=True)
Path('data/advisories.json').write_text('[\n {"id":"heat","topic":"heat","text":"When heat is extreme, avoid strenuous midday exertion, drink fluids, cool down in shade or air-conditioned spaces, and check on higher-risk people. Confusion or collapse needs urgent medical help.","source":"https://www.who.int/news-room/fact-sheets/detail/climate-change-heat-and-health"},\n {"id":"air","topic":"air_quality","text":"During particle-pollution episodes, children, older adults and people with heart or lung disease should reduce prolonged or heavy outdoor exertion.","source":"https://www.airnow.gov/aqi/aqi-basics/"},\n {"id":"heat_index","topic":"heat_index","text":"Heat index estimates how hot it feels based on air temperature and relative humidity, usually in shaded conditions; direct sunlight may increase perceived heat.","source":"https://www.weather.gov/safety/heat-index"}\n]\n', encoding="utf-8")
print("Saved", 'data/advisories.json')

In [ ]:
from pathlib import Path
Path('code/assignment_calculations.py').parent.mkdir(parents=True,exist_ok=True)
Path('code/assignment_calculations.py').write_text('#!/usr/bin/env python3\n"""Reproducible numerical work for Questions 1, 3 and 4.\n\nNo web access or API keys required. Does NOT generate or impersonate measured tool\nlatency or GPU fine-tuning results. Values are sourced in model_research.json.\n"""\nimport csv, json, math\nfrom pathlib import Path\nimport numpy as np\n\nROOT = Path(__file__).resolve().parents[1]\nDATA = ROOT / \'data\'\nRESULTS = ROOT / \'results\'\nFIG = ROOT / \'figures\'\nRESULTS.mkdir(exist_ok=True)\nFIG.mkdir(exist_ok=True)\n\nID = \'22P-9288\'\ndigits = [int(ch) for ch in ID if ch.isdigit()]\nR = int(\'\'.join(map(str,digits[-2:])))\nD = sum(digits) % 10\nQ = 1500 + 25 * R\nN = 5 + R % 4\np = .04 + .01 * D\nH = 4 + R % 4\nB = 1200 + 10 * R\nT = 10 - R % 3\nM = (30-H)*Q + H*(5*Q)\nsteps = []\nfor k in range(1,N+1):\n    t_in = 2000 + 800*(k-1)\n    t_out = 600 if k==N else 350\n    retries = p if k<N else 0.0\n    steps.append({\'k\':k,\'input\':t_in,\'output\':t_out,\'expected_retries\':retries,\n                  \'expected_input\':t_in*(1+retries),\'expected_output\':t_out*(1+retries)})\nexpected_retries = (N-1)*p\ninput_per_query = sum(x[\'expected_input\'] for x in steps)\noutput_per_query = sum(x[\'expected_output\'] for x in steps)\npeak_qpm = (5*Q)*.35/120\npeak_llm_rpm = peak_qpm*(N+expected_retries)\npeak_input_tpm = peak_qpm*input_per_query\npeak_output_tpm = peak_qpm*output_per_query\npeak_tpm = peak_input_tpm+peak_output_tpm\n# Within-query explicit-cache case: write first 1500-token call and read\n# 1500-token prefix on each subsequent call including 0.05 expected retries.\nwrite_per_query=1500\ncache_read_per_query=1500*(N-1+expected_retries)\nordinary_per_query=input_per_query - cache_read_per_query - write_per_query\nassert abs(ordinary_per_query+cache_read_per_query+write_per_query-input_per_query)<1e-6\nfacts = dict(student_id=ID,R=R,D=D,Q=Q,N=N,p=p,H=H,B=B,T=T,monthly_queries=M,\n             steps=steps,expected_retries_per_query=expected_retries,\n             expected_llm_calls=N+expected_retries,\n             input_per_query=input_per_query,output_per_query=output_per_query,\n             monthly_input=M*input_per_query,monthly_output=M*output_per_query,\n             peak_queries_per_min=peak_qpm,peak_llm_requests_per_min=peak_llm_rpm,\n             peak_input_tokens_per_min=peak_input_tpm,peak_output_tokens_per_min=peak_output_tpm,\n             peak_combined_tokens_per_min=peak_tpm,\n             cache_write_per_query=write_per_query,cache_read_per_query=cache_read_per_query,\n             cache_ordinary_per_query=ordinary_per_query)\nmodels=json.loads((DATA/\'model_research.json\').read_text(encoding=\'utf-8\'))[\'models\']\nprices=[]\nfor m in models:\n    without=M*(input_per_query*m[\'input\']+output_per_query*m[\'output\'])/1e6\n    possible=(m.get(\'cached\') is not None and m.get(\'cache_min\') is not None and m[\'cache_min\']<=1500 and m.get(\'cache_write\') is not None)\n    if possible:\n        with_cache=M*(ordinary_per_query*m[\'input\']+write_per_query*m[\'cache_write\']+\n                      cache_read_per_query*m[\'cached\']+output_per_query*m[\'output\'])/1e6\n    else: with_cache=without\n    prices.append({**m,\'cost_no_cache\':round(without,2),\'cost_with_cache\':round(with_cache,2),\n                   \'cache_case_eligible\':possible,\'within_budget_no_cache\':without<=B,\n                   \'within_budget_with_cache\':with_cache<=B})\nwith (RESULTS/\'q1_model_costs.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n    cols=[\'name\',\'kind\',\'developer\',\'provider\',\'input\',\'cached\',\'cache_write\',\'output\',\'cost_no_cache\',\'cost_with_cache\',\'cache_case_eligible\',\'within_budget_no_cache\',\'within_budget_with_cache\',\'rpm\',\'tpm\',\'ttft\',\'tps\',\'source\']\n    dw=csv.DictWriter(f,fieldnames=cols,extrasaction=\'ignore\');dw.writeheader();dw.writerows(prices)\n\n# Qwen2.5-7B-Instruct model config; given file and official Hugging Face config.\nparams=7.62e9\nvram_gb=80\nlayers,kvheads,head_dim=28,4,3584//28\nlength=5200+600 # last LLM call\'s 5,200 input + 600 answer\nkv_bytes=2*layers*kvheads*head_dim*length*2\nkv_gib=kv_bytes/(1024**3)\nweights_gib={"fp16":params*2/2**30,"8bit":params/2**30,"4bit":params*.5/2**30}\nusable_gib=.9*vram_gb\nmax_seq=math.floor((usable_gib-weights_gib[\'4bit\'])/kv_gib)\nsessions_required=math.ceil(peak_qpm*T/60)\ngpus_memory=math.ceil(sessions_required/max_seq)\n# November 10, 2026 current A100 rate from Lambda; November 16 price changes.\ngpu_hour_current=2.79  # Future-effective listed Lambda price (2026-11-16), illustrative scenario only\ngpu_hour_future=3.19  # Alternative user-supplied quotation, not verified current price\nmonthly_gpu_current=gpu_hour_current*24*30\nmonthly_gpu_future=gpu_hour_future*24*30\n# Latency redesign: deterministic routing + independent tools in parallel + ONE\n# final LLM response, bounded by original final-step 5,200 input/600 output.\n# Compare the same redesigned workload on self-hosted Qwen2.5 vs hosted Qwen2.5.\n# The 5-call original Qwen and Mistral variants violate T based on output speed;\n# after redesign Qwen is the only plausible <=9s candidate *if* tools finish fast.\ncomparator = next(x for x in models if x[\'name\']==\'Qwen2.5-7B Instruct\')\nredesign_input=5200\nredesign_output=600\nmonthly_queries_factor=30-H+5*H\nmonthly_api_per_daily_Q=monthly_queries_factor*(redesign_input*comparator[\'input\']+redesign_output*comparator[\'output\'])/1e6\n# GPU step is a MEMORY-ONLY estimate, not experimentally validated throughput.\ndef gpu_count(q):\n    rate=(5*q)*.35/120\n    concurrency=math.ceil(rate*T/60)\n    return max(1,math.ceil(concurrency/max_seq))\nq_star=None\nfor candidate in range(1,400001):\n    if gpu_count(candidate)*monthly_gpu_current < candidate*monthly_api_per_daily_Q:\n        q_star=candidate;break\nselfhosting={"gpu":"Lambda A100 80GB","gpu_hour_usd_future_effective_2026_11_16":gpu_hour_current,\n             "alternative_user_quoted_rate":gpu_hour_future,"weights_gib":weights_gib,\n             "kv_cache_gib_at_5800_tokens":kv_gib,"context_tokens":length,\n             "usable_gpu_memory_gib":usable_gib,"max_concurrent_by_memory_only":max_seq,\n             "peak_sessions_by_littles_law_9s":sessions_required,\n             "gpu_count_by_memory_only":gpus_memory,\n             "monthly_price_one_gpu_listed_future":monthly_gpu_current,\n             "monthly_price_one_gpu_alternative_quote":monthly_gpu_future,\n             "breakeven_q_memory_only_vs_qwen_redesigned":q_star,\n             "redesigned_one_call_input":redesign_input,"redesigned_one_call_output":redesign_output,\n             "redesigned_monthly_hosted_cost_at_Q":Q*monthly_api_per_daily_Q,\n             "warning":"Memory concurrency does not guarantee decode throughput. Actual GPUs needed to generate 2000+ tokens/query cannot be established without an on-GPU tokens/sec and batching benchmark."}\n# Numerical Q3\nteacher=np.array([5.,3.,2.,0.]);student=np.array([3.,2.5,1.,.5]);target=np.array([1.,0.,0.,0.])\ndef softmax(z,temp=1):\n    z=z/temp; exp=np.exp(z-z.max());return exp/exp.sum()\np1=softmax(teacher);q1=softmax(student);p2=softmax(teacher,2);q2=softmax(student,2)\nkl=float(np.sum(p2*np.log(p2/q2)))\nce=float(-math.log(q1[0]));loss=2*kl+.5*ce\ngrad=1.0*(q2-p2)+.5*(q1-target)\nnew=student-.5*grad\nnew_kl=float(np.sum(p2*np.log(p2/softmax(new,2))))\nnew_ce=float(-math.log(softmax(new)[0]));new_loss=2*new_kl+.5*new_ce\nq3={"teacher_T1":p1.tolist(),"student_T1":q1.tolist(),"teacher_T2":p2.tolist(),\n    "student_T2":q2.tolist(),"kl":kl,"ce":ce,"loss":loss,\n    "grad":grad.tolist(),"updated_logits":new.tolist(),"new_kl":new_kl,\n    "new_ce":new_ce,"new_loss":new_loss}\n\n# Q4, Wilson binomial interval to avoid extreme-normal approximation.\nmatrices={"zero_shot":np.array([[100,5,15],[8,80,12],[20,10,50]]),\n          "few_shot":np.array([[106,4,10],[6,84,10],[9,7,64]])}\nq4={};z=1.959963984540054\nfor name,matrix in matrices.items():\n    tp=np.diag(matrix); row=matrix.sum(axis=1);col=matrix.sum(axis=0);n=matrix.sum()\n    pr=tp/col; rec=tp/row; f1=2*pr*rec/(pr+rec)\n    acc=float(tp.sum()/n);den=1+z*z/n\n    center=(acc+z*z/(2*n))/den\n    half=z*math.sqrt(acc*(1-acc)/n+z*z/(4*n*n))/den\n    q4[name]={"accuracy":acc,"wilson95":[center-half,center+half],\n              "precision":pr.tolist(),"recall":rec.tolist(),"f1":f1.tolist(),\n              "macro_f1":float(f1.mean()),"weighted_f1":float(np.sum(row*f1)/n),\n              "micro_f1":acc}\nreorders=np.array([.847,.840,.853,.837,.850])\nreference=\'the delivery was late but the product quality is good\'.split()\ncandidate=\'the product quality is good but delivery was slow\'.split()\nfrom collections import Counter\n\ndef grams(seq,n):return Counter(tuple(seq[i:i+n]) for i in range(len(seq)-n+1))\nngram={}\nfor n in range(1,5):\n    c=grams(candidate,n);r=grams(reference,n)\n    overlap=sum((c&r).values());cp=sum(c.values());rp=sum(r.values())\n    ngram[n]={"overlap":overlap,"candidate_count":cp,"reference_count":rp,\n              "precision":overlap/cp if cp else 0,"recall":overlap/rp if rp else 0}\nbleu_bp=math.exp(min(0,1-len(reference)/len(candidate)))\nbleu4=bleu_bp*math.exp(sum(.25*math.log(ngram[n][\'precision\']) for n in range(1,5)))\ndef f1(p,r):return (2*p*r/(p+r)) if p+r else 0\n# LCS length as ROUGE-L sentence-level count\nL=[[0]*(len(candidate)+1) for _ in range(len(reference)+1)]\nfor i in range(1,len(reference)+1):\n    for j in range(1,len(candidate)+1):\n        L[i][j]=L[i-1][j-1]+1 if reference[i-1]==candidate[j-1] else max(L[i-1][j],L[i][j-1])\nlcs=L[-1][-1]\n# METEOR algorithm based on exact-token NLTK scorer; stemming/WordNet synonyms\n# may change score; the official NLTK score requires nltk wordnet corpus.\nmeteor_exact=None\ntry:\n    from nltk.translate.meteor_score import meteor_score\n    class NoSynsets:\n        def synsets(self, word):return []\n    meteor_exact=float(meteor_score([reference],candidate,wordnet=NoSynsets()))\nexcept Exception:\n    pass\np9=np.array([.6,.4,.5,.8,.7,.3,.25,.5,.2]);ppl=float(math.exp(-np.log(p9).mean()))\nq4[\'prompt_order\']={"mean":float(reorders.mean()),"sample_sd":float(reorders.std(ddof=1))}\nq4[\'text_metrics\']={"reference_tokens":len(reference),"candidate_tokens":len(candidate),\n    "ngrams":ngram,"bleu4_unsmoothed":bleu4,"brevity_penalty":bleu_bp,\n    "rouge1_f1":f1(ngram[1][\'precision\'],ngram[1][\'recall\']),\n    "rouge2_f1":f1(ngram[2][\'precision\'],ngram[2][\'recall\']),\n    "rougeL_lcs":lcs,"rougeL_f1":f1(lcs/len(candidate),lcs/len(reference)),\n    "meteor_nltk_no_synonyms":meteor_exact,"perplexity":ppl,\n    "perplexity_mean_nll":float(-np.log(p9).mean())}\n(RESULTS/\'verified_calculations.json\').write_text(json.dumps({"q1":facts,"selfhosting":selfhosting,\n       "q3":q3,"q4":q4},indent=2),encoding=\'utf-8\')\nwith (RESULTS/\'q1_break_even_data.csv\').open(\'w\',newline=\'\') as f:\n    w=csv.writer(f);w.writerow([\'normal_daily_queries\',\'monthly_api_qwen_redesigned_usd\',\'monthly_gpu_memory_step_usd\',\'gpu_count_memory_only\'])\n    for qq in range(500,100001,500):w.writerow([qq,qq*monthly_api_per_daily_Q,\n               gpu_count(qq)*monthly_gpu_current,gpu_count(qq)])\ntry:\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    x=np.arange(500,100001,500); api=x*monthly_api_per_daily_Q\n    gpu=np.array([gpu_count(int(qq))*monthly_gpu_current for qq in x]);\n    plt.figure(figsize=(9,5.2));plt.plot(x,api,label=\'Hosted Qwen2.5 (redesigned 1-call workflow)\',linewidth=2)\n    plt.step(x,gpu,where=\'post\',label=\'Self-hosted Qwen2.5 A100: memory-only GPU steps\',linewidth=2)\n    plt.axvline(Q,color=\'gray\',linestyle=\':\',label=f\'Assignment Q={Q}\')\n    if q_star:plt.axvline(q_star,color=\'gray\',linestyle=\'--\',label=f\'Memory-only Q*={q_star:,}\')\n    plt.xlabel(\'Normal daily queries Q\');plt.ylabel(\'Monthly cost (USD)\');plt.title(\'Cost comparison — memory-only approximation, not throughput feasibility\')\n    plt.legend(loc=\'upper left\',fontsize=8);plt.grid(alpha=.2);plt.tight_layout()\n    plt.savefig(FIG/\'q1_break_even.png\',dpi=180);plt.close()\nexcept ImportError:\n    print(\'matplotlib not installed; CSV available for manual plot\')\nprint(json.dumps({"q1":{"Q":Q,"month_queries":M,"input_per_query":input_per_query,\n     "output_per_query":output_per_query,"peak_requests_per_min":peak_llm_rpm},\n     "selfhosting":{"max_seq":max_seq,"gpus":gpus_memory,"q_star":q_star},\n     "q3_loss":loss,"q3_loss_after":new_loss,"q4_accuracy":q4[\'few_shot\'][\'accuracy\']},indent=2))\n', encoding="utf-8")
print("Saved", 'code/assignment_calculations.py')

In [ ]:
from pathlib import Path
Path('code/log_tools.py').parent.mkdir(parents=True,exist_ok=True)
Path('code/log_tools.py').write_text('#!/usr/bin/env python3\n"""Record genuine measurements for Q1. Run on TWO different Karachi dates.\n\nExamples:\n  pip install requests\n  export OPENAQ_API_KEY=\'...\'\n  python code/log_tools.py --runs 20\n  python code/log_tools.py --runs 20 --station-id 12345\n\nEach run logs OpenAQ latest PM reading, Open-Meteo weather, local heat-index,\nand local KB retrieval once. APIs are never silently substituted by fake data.\nIf no current Karachi OpenAQ station is returned, inspect the error, find a\nvalid station at explore.openaq.org and pass --station-id. All failed calls\nare logged; median uses ONLY status=ok and requires two distinct dates.\n"""\nimport argparse,csv,datetime as dt,json,os,statistics,time\nfrom pathlib import Path\nfrom zoneinfo import ZoneInfo\nimport requests\nROOT=Path(__file__).resolve().parents[1]\nOUT=ROOT/\'results\'/\'timing_log.csv\'\nKB=json.loads((ROOT/\'data\'/\'advisories.json\').read_text(encoding=\'utf-8\'))\nLAT,LON=24.8607,67.0011\ns=requests.Session()\n\ndef local_heat_index():\n    # NWS regression, degrees F, relative humidity percent; timing workload only.\n    temp_c,rh=38.0,65.0\n    t=temp_c*9/5+32\n    hi=(-42.379+2.04901523*t+10.14333127*rh-0.22475541*t*rh\n       -0.00683783*t*t-0.05481717*rh*rh+0.00122874*t*t*rh\n       +0.00085282*t*rh*rh-0.00000199*t*t*rh*rh)\n    return round((hi-32)*5/9,2)\n\ndef local_kb_lookup():\n    query=\'heat children school outdoor\';tokens=set(query.split())\n    ranked=sorted(KB,key=lambda x:len(set(x[\'text\'].lower().split())&tokens),reverse=True)\n    return ranked[0][\'id\']\n\ndef fetch(url,params=None,headers=None):\n    response=s.get(url,params=params,headers=headers,timeout=(8,20))\n    response.raise_for_status();return response.json()\n\ndef nearest_station_id(key):\n    data=fetch(\'https://api.openaq.org/v3/locations\',\n       {\'coordinates\':f\'{LAT},{LON}\',\'radius\':25000,\'limit\':100},\n       {\'X-API-Key\':key})\n    for loc in data.get(\'results\',[]):\n        names=[sensor.get(\'parameter\',{}).get(\'name\',\'\').lower() for sensor in loc.get(\'sensors\',[])]\n        if \'pm25\' in names or \'pm2.5\' in names or \'pm10\' in names:\n            return loc[\'id\']\n    raise RuntimeError(\'No Karachi station with PM2.5/PM10 in 25km; supply --station-id; do NOT claim a real reading\')\n\ndef air_quality(key,station_id):\n    data=fetch(f\'https://api.openaq.org/v3/locations/{station_id}/latest\',headers={\'X-API-Key\':key})\n    values=[record for record in data.get(\'results\',[]) if record.get(\'parameter\',{}).get(\'name\',\'\').lower() in (\'pm25\',\'pm2.5\',\'pm10\')]\n    # API can return just sensorsId in latest; this still provides real numeric values,\n    # but pollutant mapping requires sensor metadata from the location endpoint.\n    if not values:\n        sensors=fetch(f\'https://api.openaq.org/v3/locations/{station_id}/sensors\',headers={\'X-API-Key\':key}).get(\'results\',[])\n        ids={x[\'id\']:x.get(\'parameter\',{}).get(\'name\',\'\').lower() for x in sensors}\n        values=[{**v,\'parameter_name\':ids.get(v.get(\'sensorsId\'))} for v in data.get(\'results\',[]) if ids.get(v.get(\'sensorsId\')) in (\'pm25\',\'pm2.5\',\'pm10\')]\n    if not values:raise RuntimeError(f\'Station {station_id} returned no current PM2.5/PM10 measurements\')\n    record=values[0]\n    return {\'value\':record[\'value\'],\'parameter\':record.get(\'parameter_name\') or record.get(\'parameter\',{}).get(\'name\'),\n            \'utc\':record.get(\'datetime\',{}).get(\'utc\'),\'station_id\':station_id}\n\ndef weather():\n    data=fetch(\'https://api.open-meteo.com/v1/forecast\',{\n       \'latitude\':LAT,\'longitude\':LON,\n       \'current\':\'temperature_2m,relative_humidity_2m,wind_speed_10m\',\n       \'timezone\':\'Asia/Karachi\'})\n    v=data[\'current\'];return {k:v[k] for k in [\'temperature_2m\',\'relative_humidity_2m\',\'wind_speed_10m\']}\n\ndef main():\n    ap=argparse.ArgumentParser();ap.add_argument(\'--runs\',type=int,default=20);ap.add_argument(\'--station-id\',type=int)\n    args=ap.parse_args()\n    if args.runs<1:ap.error(\'--runs must be positive\')\n    key=os.environ.get(\'OPENAQ_API_KEY\')\n    if not key:raise SystemExit(\'Set OPENAQ_API_KEY in environment; never paste it in your assignment\')\n    try:station=args.station_id or nearest_station_id(key)\n    except Exception as exc:\n        print(\'Karachi station discovery failed:\',exc)\n        print(\'Will still log failed OpenAQ calls. Use --station-id for a known location.\')\n        station=None\n    OUT.parent.mkdir(parents=True,exist_ok=True)\n    with OUT.open(\'a\',newline=\'\',encoding=\'utf-8\') as fh:\n        writer=csv.writer(fh)\n        if OUT.stat().st_size==0:\n            writer.writerow([\'utc_iso\',\'karachi_date\',\'tool\',\'elapsed_seconds\',\'status\',\'detail\'])\n        jobs=[(\'openaq_pm\',lambda:air_quality(key,station) if station is not None else (_ for _ in ()).throw(ValueError(\'No Karachi PM station\'))),\n              (\'open_meteo\',weather),(\'heat_index\',local_heat_index),(\'kb_lookup\',local_kb_lookup)]\n        for i in range(args.runs):\n            for name,fn in jobs:\n                t0=time.perf_counter();status=\'ok\'\n                try:detail=json.dumps(fn(),ensure_ascii=False)\n                except Exception as exc:status=\'error\';detail=f\'{type(exc).__name__}: {exc}\'\n                elapsed=time.perf_counter()-t0\n                now=dt.datetime.now(dt.timezone.utc)\n                writer.writerow([now.isoformat(),now.astimezone(ZoneInfo(\'Asia/Karachi\')).date().isoformat(),\n                                 name,f\'{elapsed:.9f}\',status,detail[:600]])\n                fh.flush();print(f\'{i+1:02d}/{args.runs} {name:11} {elapsed:.5f}s {status}\')\n    print(\'CSV appended:\',OUT)\n    analyze()\n\ndef analyze():\n    if not OUT.exists():return\n    rows=list(csv.DictReader(OUT.open(encoding=\'utf-8\')))\n    good=[x for x in rows if x[\'status\']==\'ok\'];tools={x[\'tool\'] for x in rows}\n    print(\'Valid samples (must span two dates, >=20 per tool total):\')\n    medians=[]\n    for name in sorted(tools):\n        sub=[x for x in good if x[\'tool\']==name]\n        dates={x[\'karachi_date\'] for x in sub}\n        median=statistics.median(float(x[\'elapsed_seconds\']) for x in sub) if sub else None\n        print(f\' {name}: n={len(sub)}, dates={len(dates)}, median={median}\')\n        if len(sub)>=20 and len(dates)>=2:medians.extend(float(x[\'elapsed_seconds\']) for x in sub)\n    if len(tools)==4 and all(len([x for x in good if x[\'tool\']==name])>=20 and len({x[\'karachi_date\'] for x in good if x[\'tool\']==name})>=2 for name in tools):\n        print(\'ASSIGNMENT pooled median tool latency (seconds):\',statistics.median(medians))\n    else:print(\'NOT YET COMPLETE: need >=20 successful measurements/tool across >=2 Karachi dates.\')\n\nif __name__==\'__main__\':main()\n', encoding="utf-8")
print("Saved", 'code/log_tools.py')

## 3. Run assignment calculations
This computes R, D, Q, N, retry-adjusted token volumes, peak load, model API-cost scenarios, FP16/8-bit/4-bit memory, KV cache, capacity-based illustrative break-even, and a plot. **The GPU break-even is conditional on a memory-only capacity model.**

In [ ]:
import subprocess, sys, json
subprocess.run([sys.executable, 'code/assignment_calculations.py'],check=True)
facts=json.loads(Path('results/verified_calculations.json').read_text())
print('Workload:', {k:facts['q1'][k] for k in ['R','D','Q','N','p','H','B','T','monthly_queries','input_per_query','output_per_query','peak_llm_requests_per_min']})

In [ ]:
import pandas as pd
from IPython.display import display, Image
df=pd.read_csv('results/q1_model_costs.csv')
display(df)
if Path('figures/q1_break_even.png').exists():display(Image(filename='figures/q1_break_even.png'))

## 4. Enter the OpenAQ key securely
In Colab use **Secrets** (key icon on left) and create `OPENAQ_API_KEY`, then turn on notebook access. If unavailable, `getpass` prompts for it without printing it. Never paste a credential into a shared notebook.

In [ ]:
import os, getpass
if not os.getenv('OPENAQ_API_KEY'):
    try:
        from google.colab import userdata
        value=userdata.get('OPENAQ_API_KEY')
        if value:os.environ['OPENAQ_API_KEY']=value
    except Exception:pass
if not os.getenv('OPENAQ_API_KEY'):
    os.environ['OPENAQ_API_KEY']=getpass.getpass('OpenAQ API key (hidden): ')
print('OpenAQ key provided:',bool(os.getenv('OPENAQ_API_KEY')))

## 5. Run real four-tool timing measurements
OpenAQ: real PM sensor readings, Open-Meteo: live weather, heat index: local arithmetic, knowledge base: local guidance lookup. The supplied script saves `results/timing_log.csv` with **Karachi dates** and the error status for every call.

**Important:** 20 calls to each external API must be spread over at least two different days; run 20 samples now and repeat the same cell on a later calendar day. The CSV is appended, not overwritten. If no Karachi PM station is found automatically, inspect the API station IDs, then specify `STATION_ID`. No PM value is invented.

In [ ]:
# Set to a valid OpenAQ *location* ID only if automatic station discovery fails.
STATION_ID=None
RUNS=20
cmd=[sys.executable,'code/log_tools.py','--runs',str(RUNS)]
if STATION_ID is not None:cmd+=['--station-id',str(STATION_ID)]
subprocess.run(cmd,check=True)

In [ ]:
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo
logfile=Path('results/timing_log.csv')
if logfile.exists():
    log=pd.read_csv(logfile)
    print('Karachi date:',datetime.now(ZoneInfo('Asia/Karachi')).date())
    display(log.groupby(['tool','status']).agg(n=('elapsed_seconds','size'),dates=('karachi_date','nunique'),median_s=('elapsed_seconds','median')))
    good=log[log.status=='ok']
    totals=good.groupby('tool').agg(n=('elapsed_seconds','size'),dates=('karachi_date','nunique'))
    ready=len(totals)==4 and (totals.n>=20).all() and (totals.dates>=2).all()
    print('Measured timing evidence COMPLETE:',bool(ready))
    if ready:
        median=float(good.elapsed_seconds.median());print('Pooled median latency:',median,'seconds')
        # Expected original sequential tool delay and retry overhead.
        N=facts['q1']['N'];p=facts['q1']['p']
        print('Expected tool delay:',(N-1)*median,'seconds')
        print('Retry additional LLM calls expected:',(N-1)*p,'; failed tool-call retries do not add a second successful tool execution')
else: print('No timing log yet')

## 6. End-to-end latency for two candidates (measured inputs only)
Enter actual **TTFT in seconds** and **generation throughput in tokens/s** for two candidates in the dictionaries below. The formula includes the five sequential LLM calls, expected invalid-call retries, and four tool delays. TTFT measurements depend on prompt length and environment, so a single TTFT is a simplifying assumption. The script refuses to report a numeric end-to-end figure if genuine speed data or the validated pooled tool median is missing.

In [ ]:
import math
# Fill using cited benchmark/provider measurements; DO NOT invent figures.
CANDIDATES=[
    {'name':'Candidate A','ttft_s':None,'tokens_per_second':None,'source_url':''},
    {'name':'Candidate B','ttft_s':None,'tokens_per_second':None,'source_url':''},
]
validated_tool_median=None
if Path('results/timing_log.csv').exists():
    lg=pd.read_csv('results/timing_log.csv'); gg=lg[lg.status=='ok']
    grp=gg.groupby('tool').agg(n=('elapsed_seconds','size'),d=('karachi_date','nunique'))
    if len(grp)==4 and (grp.n>=20).all() and (grp.d>=2).all():
        validated_tool_median=float(gg.elapsed_seconds.median())
N=facts['q1']['N']; p=facts['q1']['p']; target=facts['q1']['T']
for item in CANDIDATES:
    ttft=item['ttft_s']; tps=item['tokens_per_second']
    if validated_tool_median is None or not ttft or not tps or not item['source_url']:
        print(item['name'], ': PENDING measured tool median and/or sourced TTFT, tokens/s')
        continue
    base_output=(N-1)*350+600
    retry_output=(N-1)*p*350
    model_s=(N+(N-1)*p)*ttft+(base_output+retry_output)/tps
    tools_s=(N-1)*validated_tool_median
    total=model_s+tools_s
    print(item['name'], 'model_s=',round(model_s,3),'tools_s=',round(tools_s,3),
          'total_s=',round(total,3),'meets target=',total<=target)
    print('Reference:',item['source_url'])

## 7. Export supporting evidence
Report the completed CSV, pricing table, calculation JSON, cost plot and the unmodified notebook. No Q1 training dataset is required; `data/advisories.json` is the local knowledge base and is included.

In [ ]:
import zipfile
items=['results/verified_calculations.json','results/q1_model_costs.csv','results/q1_break_even_data.csv','figures/q1_break_even.png','results/timing_log.csv','data/advisories.json','data/model_research.json']
with zipfile.ZipFile('Q1_colab_evidence.zip','w') as z:
    for item in items:
        if Path(item).exists():z.write(item)
print('Evidence archive:',ROOT/'Q1_colab_evidence.zip')

### Primary sources
- OpenAQ v3 [API documentation](https://api.openaq.org/docs) and [latest measurements](https://docs.openaq.org/resources/latest)
- Open-Meteo [Forecast API](https://open-meteo.com/en/docs)
- Lambda [GPU hourly prices](https://lambda.com/pricing) (note future-effective date)
- [Qwen2.5-7B-Instruct architecture config](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/blob/main/config.json)
- [Berkeley Function Calling Leaderboard](https://gorilla.cs.berkeley.edu/leaderboard.html)

**Additional submission checks:** screenshots of the actual model config, two full benchmark scores per model, current model-provider prices/rate limits and actual latency estimates must be checked separately. Public rates may be unavailable for a new account. Do not convert missing measurements into fabricated data.